In [3]:
import pandas as pd
from pathlib import Path


FILE_PATH = Path("online_retail.csv")

df = pd.read_csv(FILE_PATH)

print(
    df.duplicated().sum()
)

df =  df.drop_duplicates()

print(
    df.duplicated().sum()
)

print(
    df.isnull().sum()
)


df["Description"] = df[
    "Description"
].fillna("Unknown")


print(df.isnull().sum())

df = df.drop(
    columns=["CustomerID"]
)

print(
    df.isnull().sum()
)

print(
    "Negative Quantity",
    (
        df[
            "Quantity"
        ] < 0
    ).sum()
)

print(
    "Unit Price",
    (
        df[
            "UnitPrice"
        ] < 0
    ).sum()
)


print(
    df[
        df["Quantity"] < 0
    ][
        [
         "InvoiceNo",
         "StockCode",
         "Quantity",
         "InvoiceDate",
         "UnitPrice"
        ]
    ].head(20)
)

df = df[
    df[
        "Quantity"
    ] > 0
].copy()

df = df[
    df[
        "UnitPrice"
    ] > 0
].copy()

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"]
)

df["Year"] = df[
    "InvoiceDate"
].dt.year

df["Month"] = df[
    "InvoiceDate"
].dt.month

df["Week"] = df[
    "InvoiceDate"
].dt.isocalendar().week.astype(int)

df["Day_of_week"] = df[
    "InvoiceDate"
].dt.dayofweek

print(
    df.dtypes
)

df["InvoiceDate"] = df[
    "InvoiceDate"
].dt.normalize()

Daily_Demand = (
    df.groupby(
        ["InvoiceDate","StockCode"], as_index=False
    ).agg(
        Total_Quantity=(
            ("Quantity", "sum")
        )
    )
)

Daily_Demand = Daily_Demand.sort_values(
     [
         "InvoiceDate",
         "StockCode"
     ]
).reset_index(drop=True)

Daily_Demand["lag_7"] = (
    Daily_Demand.groupby("StockCode")["Total_Quantity"]
    .shift(7)
)

Daily_Demand["lag_14"] = (
    Daily_Demand.groupby("StockCode")["Total_Quantity"]
    .shift(14)
)

Daily_Demand["lag_28"] = (
    Daily_Demand.groupby("StockCode")["Total_Quantity"]
    .shift(28)
)

Daily_Demand["rolling_mean_7"] = (
    Daily_Demand.groupby("StockCode")["Total_Quantity"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

Daily_Demand["rolling_mean_14"] = (
    Daily_Demand.groupby("StockCode")["Total_Quantity"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
)

Daily_Demand["rolling_mean_28"] = (
    Daily_Demand.groupby("StockCode")["Total_Quantity"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

Daily_Demand = Daily_Demand.dropna(
    subset=[
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28"
    ]
).reset_index(drop=True)

Daily_Demand["Year"] = Daily_Demand[
    "InvoiceDate"
].dt.year

Daily_Demand["Month"] = Daily_Demand[
    "InvoiceDate"
].dt.month

Daily_Demand["Week"] = Daily_Demand[
    "InvoiceDate"
].dt.isocalendar().week.astype(int)

Daily_Demand["Day_of_week"] = Daily_Demand[
    "InvoiceDate"
].dt.dayofweek

Daily_Demand = Daily_Demand.drop(
    columns=[
        "InvoiceDate"
             ]
)


Daily_Demand.to_csv(
    "daily_demand_features.csv", index=False
)

5268
0
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64
Negative Quantity 10587
Unit Price 2
     InvoiceNo StockCode  Quantity          InvoiceDate  UnitPrice
141    C536379         D        -1  2010-12-01 09:41:00      27.50
154    C536383    35004C        -1  2010-12-01 09:49:00       4.65
235    C536391     22556       -12  2010-12-01 10:24:00       1.65
236    C536391     21984       -24  2010-12-01 10:24:00       0.29
237    C536391     21983       -24  2010-12-01 10:24:00       0.29
238    C536391     21980       -24  2010-12-